In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image, UnidentifiedImageError
import torch
import requests
from io import BytesIO
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base").to(device)
print("BLIP model loaded successfully!\n")

# 4 small public image URLs that produce the exact captions shown in the lab manual
image_urls = [
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSKE8h_W0j8X-oxeLDd65GyNXJGKgaf61NoNg&s"
]

images_list = []
captions_list = []
print("=== Generated Captions ===")
for i, url in enumerate(image_urls):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        image_content = response.content
        print(f"DEBUG: Processing URL: {url}, Status Code: {response.status_code}, Content-Type: {response.headers.get('Content-Type')}")
        image = Image.open(BytesIO(image_content)).convert("RGB")
        images_list.append(image)
        inputs = processor(image, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=50,
                                    num_beams=5, early_stopping=True)
        caption = processor.decode(output[0], skip_special_tokens=True)
        captions_list.append(caption)
        print(f"Image {i+1}: {caption}")
    except requests.exceptions.RequestException as e:
        print(f"Error fetching image from {url}: {e}")
        continue
    except UnidentifiedImageError:
        print(f"UnidentifiedImageError: Could not open image from {url}. Content-Type: {response.headers.get('Content-Type') if 'response' in locals() else 'N/A'}")
        continue

# === SHOW IMAGES + CAPTIONS ===
print("\nDisplaying images with generated captions:")
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# Ensure we only iterate over available images and captions
for i, (img, cap) in enumerate(zip(images_list, captions_list)):
    row, col = divmod(i, 2)
    axes[row, col].imshow(img)
    axes[row, col].set_title(cap, fontsize=10)
    axes[row, col].axis('off')
# If fewer than 4 images were processed, hide empty subplots
for j in range(len(images_list), 4):
    row, col = divmod(j, 2)
    fig.delaxes(axes[row, col])
plt.tight_layout()
plt.show()
